In [ ]:
include("CRD_STA.jl")
using LinearAlgebra
using NonlinearEigenproblems
using SparseArrays
using DelimitedFiles
using Plots

In [ ]:
function BF(N_cheb,Ro,Tw,Mr)
    gamma = 1.4
    sigma = 0.72
    Co = 2 - Ro - Ro^2
    u0,v0,w0,f,q,D,D2,x = baseflow_var(N_cheb,Ro,Co)
    H,T = T_ca(Mr,f,q,w0,gamma,Tw)
    F,G,H,T,rho,z = interp(u0,v0,H,T,x,N_cheb,"sim")
    lam = - (2/3) * T
    kappa = (1/sigma) * T
    return F,G,H,T,rho,x,lam,kappa,D,D2
end
function eigsol(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb,Ro,Co,D,D2,c,num)
    cof = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,N_cheb,Ro,Co,D,D2)
    L0,L1,L2 = assemble_mat(cof,D,D2,be,omega)
    nep = PEP([L0,L1,L2]); 
    eigval,eigvec = iar(nep,σ = c , neigs = num ,maxit = 500,tol=1e-12)
    # eigval = conj(eigval)
    return cof,eigval,eigvec
end

In [1071]:
N_cheb = 99
Ro = -1
Tw = 1
Mr = 0.3
gamma = 1.4
sigma = 0.72
Co = 2-Ro-Ro^2
n = 30
num = 1
omega = 0
R = 290
be = 0.077
Ma = Mr/R
F,G,H,T,rho,z,lam,kappa,D,D2 = BF(N_cheb,Ro,Tw,Mr)

([0.0; 0.0002824811191573318; … ; 9.507812671209326e-32; 9.507812671209326e-32;;], [0.0; -0.00034117928315047707; … ; -1.0; -1.0;;], [7.50356186372694e-27; -1.5650488976051362e-7; … ; -0.884474097792174; -0.884474097792174;;], [1.0; 1.0000046577842652; … ; 0.9999999853635262; 0.9999999853635262;;], [1.0; 0.9999953422374297; … ; 1.000000014636474; 1.000000014636474;;], [0.0; 0.0005539326468201291; … ; 30.0; 30.0;;], [-0.6666666666666666; -0.6666697718561767; … ; -0.6666666569090174; -0.6666666569090174;;], [1.3888888888888888; 1.3888953580337016; … ; 1.388888868560453; 1.388888868560453;;], [-2970.151515151124 3611.390030573762 … -0.9093198110834142 0.45454545454545453; -902.4262223900014 451.0994992567002 … 0.45456220808173375 -0.2272238764335357; … ; 4.436834640202506e-8 -8.875904163768224e-8 … -8.80829037817145e-5 0.00017621017590984625; -0.0 0.0 … -0.0 0.0], [5.295001504873778e6 -8.41357312149442e6 … 5384.226905905448 -2691.424943501555; 2.444105391034474e6 -3.4945616842321446e6 … 4

In [1088]:
cof = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,N_cheb,Ro,Co,D,D2)
L0_raw,L1_raw,L2_raw = assemble_mat(cof,D,D2,be,omega)
L0 = L0_raw[setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5)),setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5))]
L1 = L1_raw[setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5)),setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5))]
L2 = L2_raw[setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5)),setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5))]
nep = PEP([L0,L1,L2]); 
eigval,eigvec = iar(nep, σ = 0.4, neigs = 3,maxit = 500,tol=1e-12, logger=logger)
@show eigval
# @show logger.errs[1:20, 1]

3-element Vector{ComplexF64}:
  0.3828986388327989 - 0.0016050015807692554im
 0.22626339731472328 + 0.05174288068293109im
 0.24349626429975807 + 0.10838386783048942im

In [1096]:
vel_full,vel = eig_full(eigvec,N_cheb,3,15)

(ComplexF64[0.0 + 0.0im, 9.289762525380822e-6 - 5.336936280319758e-6im, 3.7148933953618304e-5 - 2.1144946167810617e-5im, 8.346235988976238e-5 - 4.686519522391743e-5im, 0.00014810498096242346 - 8.154471581203748e-5im, 0.00023084048114150652 - 0.0001238921950114729im, 0.00033145123818512263 - 0.00017225660150653075im, 0.0004495986421262784 - 0.00022468260765641165im, 0.000585009494267054 - 0.00027890622906565933im, 0.0007372666201073386 - 0.00033241873089152386im  …  3.957462169654289e-11 + 7.552018738013824e-11im, -2.4663751455652333e-11 - 4.731143645256286e-11im, 1.3821793210834488e-11 + 2.773327227589828e-11im, -8.086244022466327e-12 - 1.428410948550729e-11im, 3.4537472030525193e-12 + 7.364645026632088e-12im, -2.3769950851376676e-13 - 3.3854722410880944e-12im, 1.0028844887790406e-12 + 1.0073228058973815e-12im, -2.22451610868219e-12 + 6.882896239606851e-13im, 5.0638509522478515e-14 - 1.1041491271943644e-13im, 0.0 + 0.0im], (ComplexF64[0.0 + 0.0im, 9.289762525380822e-6 - 5.3369362803197

In [1090]:
L0_A_raw = L0_raw';
L1_A_raw = L1_raw';
L2_A_raw = L2_raw';
L0_A = L0_A_raw[setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5)),setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5))]
L1_A = L1_A_raw[setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5)),setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5))]
L2_A = L2_A_raw[setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5)),setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5))]
nep = PEP([L0_A,L1_A,L2_A]); 
eigval_A,eigvec_A = iar(nep,σ = 0.4 , neigs = num ,maxit = 500,tol=1e-18)
vel_full_A,vel_A = eig_full(eigvec_A,N_cheb,1,15)
@show eigval_A

1-element Vector{ComplexF64}:
 0.3828984696976057 + 0.001604867872475182im

In [1091]:
@show eigval_A,eigval

(ComplexF64[0.3828984696976057 + 0.001604867872475182im], ComplexF64[0.3828986388327989 - 0.0016050015807692554im, 0.22626339731472328 + 0.05174288068293109im, 0.24349626429975807 + 0.10838386783048942im])

In [1097]:
Q = abs(vel_full_A' * ((L1_raw + (eigval[1]+eigval_A[1]) .* L2_raw) * vel_full))

0.0014994976018795256

In [ ]:
# res_A = (L0_A +  eigval_A[1]*L1_A + eigval_A[1]^2*L2_A) * eigvec_A;
# plot(abs.(err))

In [ ]:
# plot(z,abs.(vel[1]),label = "u",xlims=[0,30],ylims=[0,0.2001]) 
# plot!(z,abs.(vel[2]),label = "v") 
# plot!(z,abs.(vel[3]),label = "w") 
# # plot(z,abs.(vel[4]),label = "rho")

In [ ]:
# plot(z,abs.(vel_A[1]),label = "u",xlims=[0,30],ylims=[0,0.2001])
# plot!(z,abs.(vel_A[2]),label = "v")
# plot!(z,abs.(vel_A[3]),label = "w")

In [1110]:
coff = [0 0]
for R = 280 : 2 : 320
    be = 28/R
    Ma = Mr/R
    cof = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,N_cheb,Ro,Co,D,D2)
    L0_raw,L1_raw,L2_raw = assemble_mat(cof,D,D2,be,omega)
    L0 = L0_raw[setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5)),setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5))]
    L1 = L1_raw[setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5)),setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5))]
    L2 = L2_raw[setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5)),setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5))]
    nep = PEP([L0,L1,L2]); 
    eigval,eigvec = iar(nep, σ = 0.4, neigs = 1,maxit = 500,tol=1e-13, logger=logger)
    vel_full,vel = eig_full(eigvec,N_cheb,1,15)
    L0_A_raw = L0_raw';
    L1_A_raw = L1_raw';
    L2_A_raw = L2_raw';
    L0_A = L0_A_raw[setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5)),setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5))]
    L1_A = L1_A_raw[setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5)),setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5))]
    L2_A = L2_A_raw[setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5)),setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5))]
    nep = PEP([L0_A,L1_A,L2_A]); 
    eigval_A,eigvec_A = iar(nep,σ = 0.4 , neigs = 1 ,maxit = 500,tol=1e-18)
    vel_full_A,vel_A = eig_full(eigvec_A,N_cheb,1,15)
    Q = abs(vel_full_A' * ((L1_raw + (eigval[1]+eigval_A[1]) .* L2_raw) * vel_full))
    x0 = R
    alpha = eigval[1]
    u_wall = -(D*F)[1] * exp(-eigval[1] * (im* x0 + 2 * eigval[1]))
    v_wall = -(D*G)[1] * exp(-eigval[1] * (im* x0 + 2 * eigval[1]))
    w_wall = 0
    vel[1][1] = u_wall
    vel[2][1] = v_wall
    vel[3][1] = w_wall
    phi = [vel[1];vel[2];vel[3];vel[4];vel[5]]
    dphi = [D*vel[1];D*vel[2];D*vel[3];D*vel[4];D*vel[5]]
    psi = [vel_A[1];vel_A[2];vel_A[3];vel_A[4];vel_A[5]]
    dpsi = [D*vel_A[1];D*vel_A[2];D*vel_A[3];D*vel_A[4];D*vel_A[5]]
    BC = (Tw/R) * (D*vel_A[1])[1] * u_wall + (3Tw/4R) * (D*vel_A[2])[1] * v_wall
    Cr = abs(-im * BC/Q)
    coff = [coff;[R Cr]]
    writedlm("Cr_lof.dat",coff[2:end,:])
end

InterruptException: InterruptException:

In [ ]:
plot(coff[2:end,1],coff[2:end,2])

In [ ]:
function integral(x,y)
    inte_progress =  zeros(ComplexF64,length(x),1)
    for i = 1 : length(x)
        inte_progress[i] = x[1:i,1]' * y[1:i,1]
    end
    return inte_progress
end

In [ ]:
function mat_diff(A,B,N_cheb)
    block_size = N_cheb + 1
    nblock = 5
    C = zeros(ComplexF64,block_size*nblock,block_size*nblock)
    for i = 0 : nblock - 1
        for j = 0 : nblock - 1
        C[1+block_size*i:block_size*(i+1),1+block_size*j:block_size*(j+1)] = (B * A[1+block_size*i:block_size*(i+1),1+block_size*j:block_size*(j+1)])
        end
    end

    return C
end

In [ ]:
function tr(A,N_cheb)
    block_size = N_cheb + 1
    nblock = 5
    C = zeros(ComplexF64,block_size*nblock,block_size*nblock)
    for i = 0 : nblock - 1
        for j = 0 : nblock - 1
            C[1+block_size*i:block_size*(i+1),1+block_size*j:block_size*(j+1)] = (A[1+block_size*i:block_size*(i+1),1+block_size*j:block_size*(j+1)])
        end
    end
    return C'
end

In [ ]:
function eig_full(eigvec,N_cheb,num,zmax)
    N = N_cheb + 1
    eigvec = eigvec[:,num]
    insert!(eigvec,5N-9,0im)
    insert!(eigvec,4N-7,0im)
    insert!(eigvec,4N-6,0im)
    insert!(eigvec,3N-5,0im)
    insert!(eigvec,3N-4,0im)
    insert!(eigvec,2N-3,0im)
    insert!(eigvec,2N-2,0im)
    insert!(eigvec,N-1,0im)
    insert!(eigvec,N,0im)
    insert!(eigvec,1,0im)
    u = eigvec[1:N,1]
    v = eigvec[N+1:2N,1]
    w = eigvec[2N+1:3N,1]
    rho = eigvec[3N+1:4N,1]
    T = eigvec[4N+1:5N,1]
    # index = findall(x->x==zmax,z)[1][1]
    # u[index:end] .= 0
    # v[index:end] .= 0
    # w[index:end] .= 0
    # rho[index:end] .= 0
    # T[index:end] .= 0
    eigvec = [u;v;w;rho;T]
    return eigvec,(u,v,w,rho,T)
end

In [ ]:
A0 = tr(cof.D1,N_cheb) + im * be * tr(cof.B,N_cheb) - im * omega * tr(cof.Ta,N_cheb) - be^2 * tr(cof.Vyy,N_cheb) - mat_diff(cof.C,D,N_cheb) - im *be*mat_diff(cof.Vyz,D,N_cheb)+ mat_diff(cof.Vzz,D2,N_cheb) - (tr(cof.C,N_cheb)+ im * be * mat_diff(cof.Vyz,D,N_cheb) - 2 * mat_diff(cof.Vzz,D,N_cheb)) * kron(I(5),D)+ tr(cof.Vzz,N_cheb) * kron(I(5),D2)
A1 = im * tr(cof.A,N_cheb) - be * tr(cof.Vxy,N_cheb) + im * mat_diff(cof.Vxz,D,N_cheb) - im * tr(cof.Vxz,N_cheb) * kron(I(5),D) 
A2 = -tr(cof.Vxx,N_cheb)
A0 = A0[setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5)),setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5))]
A1 = A1[setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5)),setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5))]
A2 = A2[setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5)),setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5))]
nep = PEP([A0,A1,A2]); 
eigval,eigvec = iar(nep,σ = 0.38, neigs = 1 ,maxit = 500,tol=1e-12)
eigval

In [ ]:
vel_full,vel = eig_full(eigvec,N_cheb,1,15)

In [ ]:
plot(z,abs.(vel[2])) 